# Paperless-ngx ML Data Platform — Chameleon Setup

Run this notebook in the **Chameleon Jupyter environment** to:
1. Provision a VM on KVM@TACC and bring up the data platform
2. Create a **persistent** object storage bucket on CHI@TACC
3. Generate S3 credentials for the bucket
4. Run the ingestion pipeline to populate the persistent bucket

# Part 1: VM Setup

## Step 1 — Select project and site

In [7]:
from chi import server, context, lease, network
import chi, os, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv("USER")
print(f"Username: {username}")

Username: dg4528_nyu_edu


## Step 2 — Reserve VM (8 hours)

In [8]:
l = lease.Lease(
    f"lease-paperless-{username}",
    duration=datetime.timedelta(hours=2)
)
l.add_flavor_reservation(id=chi.server.get_flavor_id("m1.xlarge"), amount=1)
l.submit(idempotent=True)
l.show()

Waiting for lease to start...


Lease lease-paperless-dg4528_nyu_edu has reached status active


HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>lease-paperles…

Lease Details:
Name: lease-paperless-dg4528_nyu_edu
ID: 233240f3-bf28-49f6-86cd-83e74a9de83b
Status: ACTIVE
Start Date: 2026-04-19 21:49:00
End Date: 2026-04-19 23:49:00
User ID: deebd59dc88f7570ace24af74062e708dfa95f56f54d37c9b599bde434ce37c1
Project ID: 89f528973fea4b3a981f9b2344e522de

Node Reservations:

Floating IP Reservations:

Network Reservations:

Flavor Reservations:
ID: 94f664da-596c-463f-b3e2-ff2348068aa2, Status: active, Flavor: 94f664da-596c-463f-b3e2-ff2348068aa2, Amount: 1

Events:


## Step 3 — Launch VM instance

In [9]:
s = server.Server(
    f"node-paperless-{username}",
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Waiting for server node-paperless-dg4528_nyu_edu's status to become ACTIVE. This typically takes 10 minutes for baremetal, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,node-paperless-dg4528_nyu_edu
Id,f97e2a76-28ca-44c0-ac16-95a3314f813a
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:94f664da-596c-463f-b3e2-ff2348068aa2
Addresses,sharednet1: IP: 10.56.1.210 (v4) Type: fixed MAC: fa:16:3e:d4:ca:bc
Network Name,sharednet1
Created At,2026-04-19T21:49:41Z
Keypair,dg4528_nyu_edu-jupyter
Reservation Id,None
Host Id,adf08e838260b9ebc033b298476cd6483f1fff0ef393307d1d07b094


## Step 4 — Assign floating IP

In [10]:
s.associate_floating_ip()
s.refresh()
s.show(type="widget")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Attribute,node-paperless-dg4528_nyu_edu
Id,f97e2a76-28ca-44c0-ac16-95a3314f813a
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:94f664da-596c-463f-b3e2-ff2348068aa2
Addresses,sharednet1: IP: 10.56.1.210 (v4) Type: fixed MAC: fa:16:3e:d4:ca:bc IP: 129.114.24.206 (v4) Type: floating MAC: fa:16:3e:d4:ca:bc
Network Name,sharednet1
Created At,2026-04-19T21:49:41Z
Keypair,dg4528_nyu_edu-jupyter
Reservation Id,None
Host Id,adf08e838260b9ebc033b298476cd6483f1fff0ef393307d1d07b094


## Step 5 — Open security groups

In [11]:
security_groups = [
    {"name": "allow-ssh",  "port": 22,   "description": "SSH"},
    {"name": "allow-5050", "port": 5050, "description": "Adminer UI"},
    {"name": "allow-9001", "port": 9001, "description": "MinIO Console"},
    {"name": "allow-8090", "port": 8090, "description": "Redpanda Console"},
    {"name": "allow-6333", "port": 6333, "description": "Qdrant"},
    {"name": "allow-8080", "port": 8080, "description": "Airflow UI"},
    {"name": "allow-8000", "port": 8000, "description": "Stub API"},
    {"name": "allow-5000", "port": 5000, "description": "MLflow UI"}, 
]

for sg in security_groups:
    secgroup = network.SecurityGroup({"name": sg["name"], "description": sg["description"]})
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Security groups applied: {[sg['name'] for sg in security_groups]}")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The py

Security groups applied: ['allow-ssh', 'allow-5050', 'allow-9001', 'allow-8090', 'allow-6333', 'allow-8080', 'allow-8000', 'allow-5000']


## Step 6 — Install Docker

In [12]:
s.refresh()
s.check_connectivity()
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
print("Docker installed.")

Checking connectivity to 129.114.24.206 port 22.


Connection successful


/opt/conda/lib/python3.12/site-packages/paramiko/client.py:885: UserWarning: Unknown ssh-ed25519 host key for 129.114.24.206: b'7807ade94d558241fbc2878c07cdc067'
  warnings.warn(


# Executing docker install script, commit: 8fb5881103ac6f2fb404605d6d5b1f84244f3896


+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install ca-certificates curl >/dev/null

Running kernel seems to be up-to-date.

Restarting services...
 systemctl restart packagekit.service

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on this host.
+ sh -c install -m 0755 -d /etc/apt/keyrings
+ sh -c curl -fsSL "https://download.docker.com/linux/ubuntu/gpg" -o /etc/apt/keyrings/docker.asc
+ sh -c chmod a+r /etc/apt/keyrings/docker.asc
+ sh -c echo "deb [arch=amd64 signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu noble stable" > /etc/apt/sources.list.d/docker.list
+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install docker-ce docker-ce-cli containerd.io docker-compose-plugin docker-ce-rootless-extras docker-buildx-plugin docker-model-plugin >/dev/null

Running kern

  UNIT                                                                           LOAD   ACTIVE SUB       DESCRIPTION
  proc-sys-fs-binfmt_misc.automount                                              loaded active running   Arbitrary Executable File Formats File System Automount Point
  sys-devices-pci0000:00-0000:00:03.0-virtio1-net-ens3.device                    loaded active plugged   Virtio network device
  sys-devices-pci0000:00-0000:00:04.0-virtio2-block-vda-vda1.device              loaded active plugged   /sys/devices/pci0000:00/0000:00:04.0/virtio2/block/vda/vda1
  sys-devices-pci0000:00-0000:00:04.0-virtio2-block-vda-vda2.device              loaded active plugged   /sys/devices/pci0000:00/0000:00:04.0/virtio2/block/vda/vda2
  sys-devices-pci0000:00-0000:00:04.0-virtio2-block-vda-vda3.device              loaded active plugged   /sys/devices/pci0000:00/0000:00:04.0/virtio2/block/vda/vda3
  sys-devices-pci0000:00-0000:00:04.0-virtio2-block-vda.device                   loaded active

INFO: Docker daemon enabled and started

+ 

-serial8250:0.14-tty-ttyS14.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.14/tty/ttyS14
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.15-tty-ttyS15.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.15/tty/ttyS15
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.16-tty-ttyS16.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.16/tty/ttyS16
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.17-tty-ttyS17.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.17/tty/ttyS17
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.18-tty-ttyS18.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.18/tty/ttyS18
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.19-tty-ttyS19.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/seria

sh -c docker version


l8250:0.19/tty/ttyS19
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.2-tty-ttyS2.device   loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.2/tty/ttyS2
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.20-tty-ttyS20.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.20/tty/ttyS20
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.21-tty-ttyS21.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.21/tty/ttyS21
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.22-tty-ttyS22.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.22/tty/ttyS22
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.23-tty-ttyS23.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.23/tty/ttyS23
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.24-tty-ttyS24.device loaded ac

## Step 7 — Clone repo and bring up the platform

In [ ]:
REPO_URL = "https://github.com/REDES01/paperless_data.git"

s.execute(f"git clone {REPO_URL} ~/paperless_data")
s.execute("cd ~/paperless_data && sg docker -c 'docker compose -f docker/docker-compose.yaml up -d'")
print("Platform started.")

## Step 8 — Print access URLs

In [ ]:
s.refresh()
addresses = s.addresses
floating_ip = None
for net, addrs in addresses.items():
    for addr in addrs:
        if addr.get("OS-EXT-IPS:type") == "floating":
            floating_ip = addr["addr"]

print(f"Floating IP: {floating_ip}")
print(f"")
print(f"  MinIO Console   ->  http://{floating_ip}:9001  (admin / paperless_minio)")
print(f"  Adminer (PG)    ->  http://{floating_ip}:5050  (user / paperless_postgres)")
print(f"  Redpanda UI     ->  http://{floating_ip}:8090")
print(f"  Qdrant          ->  http://{floating_ip}:6333/dashboard")
print(f"")
print(f"  SSH: ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")

---
# Part 2: Persistent Object Storage (CHI@TACC)

The VM's local MinIO is ephemeral — it dies when the VM is deleted.
For the persistent storage deliverable, we use **CHI@TACC object storage**,
which is Chameleon's managed S3-compatible store that persists independently of VMs.

## Step 9 — Generate S3 credentials for CHI@TACC

In [ ]:
from openstack import connection
from chi import context
import chi

context.choose_project()
context.choose_site(default="CHI@TACC")

In [ ]:
conn = chi.clients.connection()

project_id = conn.current_project_id
identity_ep = conn.session.get_endpoint(service_type="identity", interface="public")
url = f"{identity_ep}/v3/users/{conn.current_user_id}/credentials/OS-EC2"

resp = conn.session.post(url, json={"tenant_id": project_id})
resp.raise_for_status()
ec2 = resp.json()["credential"]

chi_access_key = ec2["access"]
chi_secret_key = ec2["secret"]

print(f"EC2 Access: {chi_access_key}")
print(f"EC2 Secret: {chi_secret_key}")
print()
print("SAVE THESE — you will need them for the make targets.")

## Step 10 — Create the persistent bucket

Create the object storage container (bucket) at CHI@TACC.
This bucket persists even if the VM is deleted.

In [ ]:
CHI_BUCKET = f"paperless-chi-{username}"

os_conn = chi.clients.connection()
token = os_conn.authorize()
storage_url = os_conn.object_store.get_endpoint()

import swiftclient
swift_conn = swiftclient.Connection(
    preauthurl=storage_url,
    preauthtoken=token,
    retries=5
)

swift_conn.put_container(CHI_BUCKET)
print(f"Created persistent bucket: {CHI_BUCKET}")
print(f"Browse it in Horizon: https://chi.tacc.chameleoncloud.org > Object Store > Containers")

## Step 11 — Set credentials on the VM

Export the S3 credentials on the VM so `make chi-ingest` can use them.

In [ ]:
# Switch back to KVM@TACC to access the VM
context.choose_site(default="KVM@TACC")
s = server.Server(f"node-paperless-{username}", image_name="CC-Ubuntu24.04", flavor_name="m1.xlarge")
s.submit(idempotent=True)

# Write credentials to a file on the VM for make targets to use
s.execute(f"""cat > ~/paperless_data/.env.chi << 'EOF'
export CHI_ACCESS_KEY={chi_access_key}
export CHI_SECRET_KEY={chi_secret_key}
export CHI_BUCKET={CHI_BUCKET}
EOF
""")
print("Credentials written to ~/paperless_data/.env.chi")
print()
print("To run the persistent ingestion pipeline, SSH in and run:")
print(f"  cd ~/paperless_data")
print(f"  source .env.chi")
print(f"  make chi-ingest")

## Step 12 — Run ingestion to persistent storage

This uploads IAM + SQuAD datasets directly to CHI@TACC persistent storage.
Course staff can browse the bucket in the Horizon GUI without needing VM access.

In [ ]:
s.execute(f"cd ~/paperless_data && source .env.chi && sg docker -c 'make chi-ingest-iam'")
print("IAM ingestion to CHI@TACC complete.")

In [ ]:
s.execute(f"cd ~/paperless_data && source .env.chi && sg docker -c 'make chi-ingest-squad'")
print("SQuAD ingestion to CHI@TACC complete.")

In [ ]:
s.execute(f"cd ~/paperless_data && source .env.chi && sg docker -c 'make chi-augment-iam'")
print("IAM augmentation to CHI@TACC complete.")

## Step 13 — Verify in Horizon

Open the CHI@TACC Horizon GUI:
1. Go to https://chi.tacc.chameleoncloud.org
2. Navigate to **Object Store > Containers**
3. Click on your bucket (e.g., `paperless-chi-xz5039-nyu-edu`)
4. You should see `warehouse/iam_dataset/` and `warehouse/squad_dataset/` with Parquet files

---
# Teardown

Run this when you are done to release VM resources.

**Note:** The CHI@TACC bucket is NOT deleted here — it persists for grading.

In [ ]:
# Uncomment and run when done
# context.choose_site(default="KVM@TACC")
# s = server.get_server(f"node-paperless-{username}")
# server.delete_server(s.id)
# l = lease.get_lease(f"lease-paperless-{username}")
# lease.delete_lease(l.id)
# print("VM resources released. CHI@TACC bucket still available for grading.")